# ⏩ Fluxuri de lucru secvențiale cu agenți folosind Microsoft Foundry (Python)

## 📋 Tutorial Avansat de Procesare Secvențială

Acest notebook demonstrează **modele de fluxuri de lucru secvențiale** utilizând Microsoft Agent Framework. Veți învăța cum să construiți conducte sofisticate de procesare în mai mulți pași, în care agenții execută într-o ordine specifică, transferând date și context între etape.

> **Notă despre migrare:** Acest exemplu făcea anterior referire la Modelele GitHub. Modelele GitHub sunt depreciate (se retrag în iulie 2026), așa că acum utilizează **Microsoft Foundry** prin `FoundryChatClient`, care țintește API-ul Azure OpenAI **Responses API**.

## 🎯 Obiective de Învățare

### 🔄 **Modele de procesare secvențială**
- **Design de flux liniar**: Crearea conductelor de procesare pas cu pas
- **Gestionarea fluxului de date**: Transferul informațiilor între agenții secvențiali
- **Procesarea tip stage-gate**: Implementarea punctelor de control și a etapelor de validare
- **Monitorizarea progresului**: Urmărirea execuției fluxului de lucru și a rezultatelor intermediare

### 🏗️ **Arhitectura conductelor pentru întreprinderi**
- **Modelarea proceselor de afaceri**: Cartografierea proceselor reale de afaceri în fluxurile de lucru ale agenților
- **Asigurarea calității**: Procese de validare și revizuire în mai multe etape
- **Procesarea documentelor**: Analiză și transformare secvențială a documentelor
- **Producția de conținut**: Fluxuri editoriale cu etape de revizuire și aprobare

### 📊 **Funcționalități avansate ale fluxului de lucru**
- **Păstrarea contextului**: Menținerea stării pe parcursul etapelor fluxului de lucru
- **Propagarea erorilor**: Gestionarea eșecurilor în procesarea secvențială
- **Optimizarea performanței**: Modele eficiente de execuție secvențială
- **Urmărirea auditului**: Monitorizarea completă a operațiunilor secvențiale

## ⚙️ Cerințe și Configurare

### 📦 **Dependențe**
```bash
pip install agent-framework -U
```

### 🔑 **Configurație**

Autentificați-vă cu Azure CLI (`az login`) pentru ca `AzureCliCredential` să se poată autentifica, apoi setați detaliile proiectului Microsoft Foundry.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

## 🏢 **Cazuri de utilizare ale fluxurilor de lucru secvențiale pentru întreprinderi**

### 📝 **Conducta de procesare a documentelor**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Flux de lucru pentru asigurarea calității** 
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Conducta de producție a conținutului**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Automatizarea proceselor de afaceri**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Principii de design pentru fluxuri de lucru secvențiale**

- **🔗 Progresie liniară**: Fiecare etapă depinde de rezultatul etapei anterioare
- **📋 Gestionarea stării**: Păstrarea contextului și a datelor pe toate etapele
- **🛡️ Gestionarea erorilor**: Gestionarea elegantă a eșecurilor în orice etapă
- **📊 Monitorizarea progresului**: Urmărirea finalizării și performanței fiecărei etape
- **🔄 Reutilizarea etapelor**: Proiectarea de componente reutilizabile ale fluxului de lucru

Să construim fluxuri de procesare secvențiale sofisticate! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
